# Python practice 8: files, paths and formats

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-08-files-paths-and-formats.ipynb)

14 exercises on paths, file I/O, atomic writes, CSV, JSON, gzip, configuration and SQLite. Each exercise names the [reference entries](../learning/python-reference-08-files-paths-and-formats.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 6 easy, 6 medium, 2 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Hive-style partition paths](#exercise-1-hive-style-partition-paths) | Easy | [Building and taking apart paths](../learning/python-reference-08-files-paths-and-formats.ipynb#building-and-taking-apart-paths) |
| 2 | [Strip every suffix](#exercise-2-strip-every-suffix) | Easy | [Building and taking apart paths](../learning/python-reference-08-files-paths-and-formats.ipynb#building-and-taking-apart-paths) |
| 3 | [Find files recursively](#exercise-3-find-files-recursively) | Easy | [Filesystem operations](../learning/python-reference-08-files-paths-and-formats.ipynb#filesystem-operations) |
| 4 | [Count records without loading the file](#exercise-4-count-records-without-loading-the-file) | Easy | [`open()`](../learning/python-reference-08-files-paths-and-formats.ipynb#open) |
| 5 | [Atomic state file](#exercise-5-atomic-state-file) | Medium | [Temporary files, `shutil` and atomic writes](../learning/python-reference-08-files-paths-and-formats.ipynb#temporary-files-shutil-and-atomic-writes) |
| 6 | [Typed CSV parsing](#exercise-6-typed-csv-parsing) | Medium | [`csv`](../learning/python-reference-08-files-paths-and-formats.ipynb#csv) |
| 7 | [Write CSV correctly](#exercise-7-write-csv-correctly) | Easy | [`csv`](../learning/python-reference-08-files-paths-and-formats.ipynb#csv) |
| 8 | [JSON Lines with bad records](#exercise-8-json-lines-with-bad-records) | Medium | [`json` and JSON Lines](../learning/python-reference-08-files-paths-and-formats.ipynb#json-and-json-lines) |
| 9 | [Serialise rich Python values](#exercise-9-serialise-rich-python-values) | Easy | [`json` and JSON Lines](../learning/python-reference-08-files-paths-and-formats.ipynb#json-and-json-lines) |
| 10 | [Gzipped JSON Lines round trip](#exercise-10-gzipped-json-lines-round-trip) | Medium | [Compressed files: `gzip`, `zipfile`](../learning/python-reference-08-files-paths-and-formats.ipynb#compressed-files-gzip-zipfile) |
| 11 | [Aggregate with SQLite](#exercise-11-aggregate-with-sqlite) | Medium | [Querying SQLite from Python](../learning/python-reference-08-files-paths-and-formats.ipynb#querying-sqlite-from-python) |
| 12 | [Idempotent upserts](#exercise-12-idempotent-upserts) | Hard | [Querying SQLite from Python](../learning/python-reference-08-files-paths-and-formats.ipynb#querying-sqlite-from-python) |
| 13 | [Layered TOML configuration](#exercise-13-layered-toml-configuration) | Medium | [Configuration: `tomllib`, `configparser`, environment variables](../learning/python-reference-08-files-paths-and-formats.ipynb#configuration-tomllib-configparser-environment-variables) |
| 14 | [Union CSV files with different columns](#exercise-14-union-csv-files-with-different-columns) | Hard | [`csv`](../learning/python-reference-08-files-paths-and-formats.ipynb#csv), [Filesystem operations](../learning/python-reference-08-files-paths-and-formats.ipynb#filesystem-operations) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction


import json
import tempfile
from pathlib import Path


def make_tree(files):
    """Create a fresh temporary directory holding {relative_path: text} and return its Path."""
    root = Path(tempfile.mkdtemp())
    for relative, text in files.items():
        path = root / relative
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
    return root

## Exercise 1: Hive-style partition paths

**Difficulty:** Easy · about 5 min · **Practises:** [Building and taking apart paths](../learning/python-reference-08-files-paths-and-formats.ipynb#building-and-taking-apart-paths)

*Lakes lay out files by `key=value` folders so engines can prune partitions.*

Write `partition_key(base, table, day)` taking a `date` and returning the POSIX string `<base>/<table>/year=YYYY/month=MM/day=DD/part-0000.parquet` using `PurePosixPath`.

**Sample calls**

| call | expected |
| --- | --- |
| `partition_key("warehouse", "orders", date(2026, 1, 5))` | `'warehouse/orders/year=2026/month=01/day=05/part-0000.parquet'` |
| `partition_key("lake/raw", "events", date(2026, 11, 27))` | `'lake/raw/events/year=2026/month=11/day=27/part-0000.parquet'` |
| `partition_key("lake/", "t", date(2026, 12, 31))` | `'lake/t/year=2026/month=12/day=31/part-0000.parquet'` |

In [ ]:
def partition_key(base, table, day):
    # your code here
    pass

In [ ]:
from pathlib import PurePosixPath

def partition_key(base, table, day):
    return str(PurePosixPath(base) / table / f"year={day:%Y}" / f"month={day:%m}" / f"day={day:%d}" / "part-0000.parquet")

## Exercise 2: Strip every suffix

**Difficulty:** Easy · about 5 min · **Practises:** [Building and taking apart paths](../learning/python-reference-08-files-paths-and-formats.ipynb#building-and-taking-apart-paths)

*Files like `orders.csv.gz` need their base name for the output table.*

Write `base_name(path)` returning the file name with all suffixes removed (`orders.csv.gz` → `orders`), and `output_path(path, suffix)` returning the same directory with that base name plus `suffix`, as a POSIX string.

**Sample calls**

| call | expected |
| --- | --- |
| `base_name("raw/orders.csv.gz")` | `'orders'` |
| `output_path("raw/orders.csv.gz", ".parquet")` | `'raw/orders.parquet'` |
| `base_name("data/README")` | `'README'` |

In [ ]:
def base_name(path):
    # your code here
    pass


def output_path(path, suffix):
    # your code here
    pass

In [ ]:
from pathlib import PurePosixPath

def base_name(path):
    p = PurePosixPath(path)
    return p.name[: len(p.name) - len("".join(p.suffixes))] if p.suffixes else p.name

def output_path(path, suffix):
    p = PurePosixPath(path)
    return str(p.parent / (base_name(path) + suffix))

## Exercise 3: Find files recursively

**Difficulty:** Easy · about 5 min · **Practises:** [Filesystem operations](../learning/python-reference-08-files-paths-and-formats.ipynb#filesystem-operations)

*Listing input files for a batch job.*

Write `find_files(root, pattern)` returning the sorted POSIX paths, relative to `root`, of files under `root` matching the glob `pattern` at any depth. Directories are excluded. `make_tree` in the imports cell creates test folders.

**Sample calls**

| call | expected |
| --- | --- |
| `find_files(make_tree({"a/1.csv": "", "a/b/2.csv": "", "c.txt": ""}), "*.csv")` | `['a/1.csv', 'a/b/2.csv']` |
| `find_files(make_tree({"x.txt": ""}), "*.csv")` | `[]` |
| `find_files(make_tree({"z.csv": "", "a.csv": ""}), "*.csv")` | `['a.csv', 'z.csv']` |

In [ ]:
def find_files(root, pattern):
    # your code here
    pass

In [ ]:
from pathlib import Path

def find_files(root, pattern):
    root = Path(root)
    return sorted(p.relative_to(root).as_posix() for p in root.rglob(pattern) if p.is_file())

## Exercise 4: Count records without loading the file

**Difficulty:** Easy · about 5 min · **Practises:** [`open()`](../learning/python-reference-08-files-paths-and-formats.ipynb#open)

*Row counts for reconciliation on files bigger than memory.*

Write `count_records(path)` returning the number of non-blank lines, reading the file line by line with `encoding="utf-8"`.

**Sample calls**

| call | expected |
| --- | --- |
| `count_records(make_tree({"f": "a\nb\nc\n"}) / "f")` | `3` |
| `count_records(make_tree({"f": "a\n\n  \nb"}) / "f")` | `2` |
| `count_records(make_tree({"f": ""}) / "f")` | `0` |

In [ ]:
def count_records(path):
    # your code here
    pass

In [ ]:
def count_records(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## Exercise 5: Atomic state file

**Difficulty:** Medium · about 10 min · **Practises:** [Temporary files, `shutil` and atomic writes](../learning/python-reference-08-files-paths-and-formats.ipynb#temporary-files-shutil-and-atomic-writes)

*Watermark and checkpoint files must never be left half-written by a crash.*

Write `save_state(path, state)` writing `state` as JSON (sorted keys) atomically: write a temp file in the same directory, then `os.replace`. Return the path as a string. No temp files may be left behind.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda d: (save_state(d / "s.json", {"b": 2, "a": 1}), (d / "s.json").read_text())[1])(make_tree({}))` | `'{"a": 1, "b": 2}'` |
| `(lambda d: (save_state(d / "s.json", {"v": 1}), save_state(d / "s.json", {"v": 2}), json.loads((d / "s.json").read_text()))[2])(make_tree({}))` | `{'v': 2}` |
| `(lambda d: (save_state(d / "s.json", {}), sorted(p.name for p in d.iterdir()))[1])(make_tree({}))` | `['s.json']` |

In [ ]:
def save_state(path, state):
    # your code here
    pass

In [ ]:
import json
import os
import tempfile
from pathlib import Path

def save_state(path, state):
    path = Path(path)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, delete=False, suffix=".tmp") as tmp:
        json.dump(state, tmp, sort_keys=True)
    os.replace(tmp.name, path)
    return str(path)

## Exercise 6: Typed CSV parsing

**Difficulty:** Medium · about 10 min · **Practises:** [`csv`](../learning/python-reference-08-files-paths-and-formats.ipynb#csv)

*CSV gives you strings; loaders must convert types and nulls.*

Write `read_typed(text, types)` parsing CSV text with a header. `types` maps column → converter (`int`, `float`, ...); unlisted columns stay strings; empty strings become None for every column.

**Sample calls**

| call | expected |
| --- | --- |
| `read_typed("id,amount,name\n1,9.5,a\n", {"id": int, "amount": float})` | `[{'id': 1, 'amount': 9.5, 'name': 'a'}]` |
| `read_typed("id,amount\n2,\n", {"id": int, "amount": float})` | `[{'id': 2, 'amount': None}]` |
| `read_typed('id,name\n1,"Smith, J"\n', {"id": int})` | `[{'id': 1, 'name': 'Smith, J'}]` |

In [ ]:
def read_typed(text, types):
    # your code here
    pass

In [ ]:
import csv
import io

def read_typed(text, types):
    rows = []
    for row in csv.DictReader(io.StringIO(text)):
        rows.append({k: None if v == "" else types.get(k, str)(v) for k, v in row.items()})
    return rows

## Exercise 7: Write CSV correctly

**Difficulty:** Easy · about 5 min · **Practises:** [`csv`](../learning/python-reference-08-files-paths-and-formats.ipynb#csv)

*Hand-joined CSV breaks on commas and quotes in values.*

Write `to_csv(rows, columns)` returning CSV text with a header row, `\n` line endings, columns in the given order (missing keys empty, None empty).

**Sample calls**

| call | expected |
| --- | --- |
| `to_csv([{"id": 1, "name": "a"}], ["id", "name"])` | `'id,name\n1,a\n'` |
| `to_csv([{"name": 'Smith, "J"'}], ["name"])` | `'name\n"Smith, ""J"""\n'` |
| `to_csv([{"id": None}], ["id", "x"])` | `'id,x\n,\n'` |

In [ ]:
def to_csv(rows, columns):
    # your code here
    pass

In [ ]:
import csv
import io

def to_csv(rows, columns):
    out = io.StringIO()
    writer = csv.DictWriter(out, fieldnames=columns, lineterminator="\n", extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)
    return out.getvalue()

## Exercise 8: JSON Lines with bad records

**Difficulty:** Medium · about 10 min · **Practises:** [`json` and JSON Lines](../learning/python-reference-08-files-paths-and-formats.ipynb#json-and-json-lines)

*Event files contain the odd corrupt line; loaders quarantine it instead of failing the batch.*

Write `parse_jsonl(text)` returning `(records, bad_lines)` where records are the parsed objects of valid non-blank lines and bad_lines lists 1-based line numbers that failed to parse or are not JSON objects.

**Sample calls**

| call | expected |
| --- | --- |
| `parse_jsonl('{"id": 1}\nnot json\n{"id": 2}\n')` | `([{'id': 1}, {'id': 2}], [2])` |
| `parse_jsonl('\n{"a": 1}\n\n')` | `([{'a': 1}], [])` |
| `parse_jsonl("")` | `([], [])` |

In [ ]:
def parse_jsonl(text):
    # your code here
    pass

In [ ]:
import json

def parse_jsonl(text):
    records, bad = [], []
    for number, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        try:
            value = json.loads(line)
        except json.JSONDecodeError:
            bad.append(number)
            continue
        if isinstance(value, dict):
            records.append(value)
        else:
            bad.append(number)
    return records, bad

## Exercise 9: Serialise rich Python values

**Difficulty:** Easy · about 5 min · **Practises:** [`json` and JSON Lines](../learning/python-reference-08-files-paths-and-formats.ipynb#json-and-json-lines)

*Exporting records with Decimal and datetime values to JSON.*

Write `to_json(record)` returning compact JSON (`separators=(",", ":")`, sorted keys, non-ASCII kept) where `Decimal` becomes a string, `date`/`datetime` ISO strings and sets sorted lists; other unsupported types raise `TypeError`.

**Sample calls**

| call | expected |
| --- | --- |
| `to_json({"amount": Decimal("1.10"), "id": 1})` | `'{"amount":"1.10","id":1}'` |
| `to_json({"d": date(2026, 1, 2), "t": datetime(2026, 1, 2, 3, 4)})` | `'{"d":"2026-01-02","t":"2026-01-02T03:04:00"}'` |
| `to_json({"tags": {"b", "a"}, "city": "Zürich"})` | `'{"city":"Zürich","tags":["a","b"]}'` |

In [ ]:
def to_json(record):
    # your code here
    pass

In [ ]:
import json
from datetime import date
from decimal import Decimal

def _default(value):
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"not JSON serialisable: {type(value).__name__}")

def to_json(record):
    return json.dumps(record, default=_default, sort_keys=True, ensure_ascii=False, separators=(",", ":"))

## Exercise 10: Gzipped JSON Lines round trip

**Difficulty:** Medium · about 10 min · **Practises:** [Compressed files: `gzip`, `zipfile`](../learning/python-reference-08-files-paths-and-formats.ipynb#compressed-files-gzip-zipfile)

*Event archives are usually gzipped JSON Lines.*

Write `write_gz(path, records)` writing one JSON object per line into a gzip file (text mode, UTF-8), returning the record count, and `read_gz(path)` returning the list of records.

**Sample calls**

| call | expected |
| --- | --- |
| `(lambda p: (write_gz(p, [{"id": 1}, {"id": 2}]), read_gz(p)))(make_tree({}) / "e.jsonl.gz")` | `(2, [{'id': 1}, {'id': 2}])` |
| `(lambda p: (write_gz(p, [{}]), p.read_bytes()[:2])[1])(make_tree({}) / "e.gz")` | `b'\x1f\x8b'` |
| `(lambda p: (write_gz(p, []), read_gz(p)))(make_tree({}) / "e.gz")` | `(0, [])` |

In [ ]:
def write_gz(path, records):
    # your code here
    pass


def read_gz(path):
    # your code here
    pass

In [ ]:
import gzip
import json

def write_gz(path, records):
    count = 0
    with gzip.open(path, "wt", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    return count

def read_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

## Exercise 11: Aggregate with SQLite

**Difficulty:** Medium · about 10 min · **Practises:** [Querying SQLite from Python](../learning/python-reference-08-files-paths-and-formats.ipynb#querying-sqlite-from-python)

*Using SQL for a quick aggregation over Python rows.*

Write `region_totals(rows)` loading `(order_id, region, amount)` rows into an in-memory SQLite table with `executemany` and returning `[(region, order_count, total_amount), ...]` ordered by total descending then region.

**Sample calls**

| call | expected |
| --- | --- |
| `region_totals([(1, "n", 120.0), (2, "s", 70.0), (3, "n", 50.0)])` | `[('n', 2, 170.0), ('s', 1, 70.0)]` |
| `region_totals([])` | `[]` |
| `region_totals([(1, "b", 5.0), (2, "a", 5.0)])` | `[('a', 1, 5.0), ('b', 1, 5.0)]` |

In [ ]:
def region_totals(rows):
    # your code here
    pass

In [ ]:
import sqlite3

def region_totals(rows):
    conn = sqlite3.connect(":memory:")
    try:
        conn.execute("CREATE TABLE orders (order_id INTEGER, region TEXT, amount REAL)")
        conn.executemany("INSERT INTO orders VALUES (?, ?, ?)", rows)
        query = "SELECT region, COUNT(*), SUM(amount) FROM orders GROUP BY region ORDER BY SUM(amount) DESC, region"
        return conn.execute(query).fetchall()
    finally:
        conn.close()

## Exercise 12: Idempotent upserts

**Difficulty:** Hard · about 15 min · **Practises:** [Querying SQLite from Python](../learning/python-reference-08-files-paths-and-formats.ipynb#querying-sqlite-from-python)

*Loads must be safe to re-run: upserting by key instead of appending duplicates.*

Write `apply_batches(batches)` creating an in-memory table `customers(id INTEGER PRIMARY KEY, email TEXT, updated INTEGER)`, applying each batch of `(id, email, updated)` rows in its own transaction with `INSERT ... ON CONFLICT(id) DO UPDATE` that only overwrites when the incoming `updated` is newer, and returning all rows ordered by id.

**Sample calls**

| call | expected |
| --- | --- |
| `apply_batches([[(1, "a@x", 1)], [(1, "b@x", 2)]])` | `[(1, 'b@x', 2)]` |
| `apply_batches([[(1, "new@x", 5)], [(1, "old@x", 3)]])` | `[(1, 'new@x', 5)]` |
| `(lambda b: apply_batches(b) == apply_batches(b + b))([[(1, "a", 1), (2, "b", 1)]])` | `True` |

In [ ]:
def apply_batches(batches):
    # your code here
    pass

In [ ]:
import sqlite3

def apply_batches(batches):
    conn = sqlite3.connect(":memory:")
    try:
        conn.execute("CREATE TABLE customers (id INTEGER PRIMARY KEY, email TEXT, updated INTEGER)")
        sql = (
            "INSERT INTO customers VALUES (?, ?, ?) ON CONFLICT(id) DO UPDATE SET "
            "email = excluded.email, updated = excluded.updated WHERE excluded.updated > customers.updated"
        )
        for batch in batches:
            with conn:
                conn.executemany(sql, batch)
        return conn.execute("SELECT id, email, updated FROM customers ORDER BY id").fetchall()
    finally:
        conn.close()

## Exercise 13: Layered TOML configuration

**Difficulty:** Medium · about 10 min · **Practises:** [Configuration: `tomllib`, `configparser`, environment variables](../learning/python-reference-08-files-paths-and-formats.ipynb#configuration-tomllib-configparser-environment-variables)

*Job configs combine defaults, a TOML file and environment overrides.*

Write `load_config(toml_text, defaults, env)` returning a dict: nested `defaults` deep-merged with the parsed TOML (TOML wins; nested tables merge), then any `env` key of the form `SECTION__KEY` (upper case) overrides `config[section][key]` with the env string converted to the type of the existing value (int, float, bool from `true`/`false`, else str).

**Sample calls**

| call | expected |
| --- | --- |
| `load_config("[job]\nretries = 5\n", {"job": {"retries": 3, "name": "x"}}, {})` | `{'job': {'retries': 5, 'name': 'x'}}` |
| `load_config("", {"job": {"retries": 3, "debug": False, "rate": 0.5}}, {"JOB__RETRIES": "7", "JOB__DEBUG": "true", "JOB__RATE": "1.5"})` | `{'job': {'retries': 7, 'debug': True, 'rate': 1.5}}` |
| `load_config("[source]\npath = \"s3://raw\"\n", {"job": {}}, {})` | `{'job': {}, 'source': {'path': 's3://raw'}}` |

In [ ]:
def load_config(toml_text, defaults, env):
    # your code here
    pass

In [ ]:
import copy
import tomllib

def _merge(base, override):
    out = copy.deepcopy(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = _merge(out[key], value)
        else:
            out[key] = value
    return out

def _convert(text, like):
    if isinstance(like, bool):
        return text.lower() == "true"
    if isinstance(like, int):
        return int(text)
    if isinstance(like, float):
        return float(text)
    return text

def load_config(toml_text, defaults, env):
    config = _merge(defaults, tomllib.loads(toml_text))
    for name, raw in env.items():
        section, sep, key = name.lower().partition("__")
        if sep and isinstance(config.get(section), dict):
            config[section][key] = _convert(raw, config[section].get(key, raw))
    return config

## Exercise 14: Union CSV files with different columns

**Difficulty:** Hard · about 20 min · **Practises:** [`csv`](../learning/python-reference-08-files-paths-and-formats.ipynb#csv), [Filesystem operations](../learning/python-reference-08-files-paths-and-formats.ipynb#filesystem-operations)

*Daily drops gain columns over time; the loader must union them without losing data.*

Write `union_csvs(root)` reading every `*.csv` under `root` (sorted by relative path) and returning `(columns, rows)`: columns in first-seen order plus a final `_source` column, and rows as lists aligned to those columns with None for missing values and `_source` set to the file's relative POSIX path.

**Sample calls**

| call | expected |
| --- | --- |
| `union_csvs(make_tree({"d1.csv": "id,a\n1,x\n", "d2.csv": "id,b\n2,y\n"}))` | `(['id', 'a', 'b', '_source'], [['1', 'x', None, 'd1.csv'], ['2', None, 'y', 'd2.csv']])` |
| `union_csvs(make_tree({"b/1.csv": "id\n2\n", "a/1.csv": "id\n1\n"}))` | `(['id', '_source'], [['1', 'a/1.csv'], ['2', 'b/1.csv']])` |
| `union_csvs(make_tree({}))` | `(['_source'], [])` |

In [ ]:
def union_csvs(root):
    # your code here
    pass

In [ ]:
import csv
from pathlib import Path

def union_csvs(root):
    root = Path(root)
    columns, records = [], []
    for path in sorted(root.rglob("*.csv"), key=lambda p: p.relative_to(root).as_posix()):
        with open(path, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for name in reader.fieldnames or []:
                if name not in columns:
                    columns.append(name)
            source = path.relative_to(root).as_posix()
            records.extend((row, source) for row in reader)
    rows = [[row.get(c) for c in columns] + [source] for row, source in records]
    return columns + ["_source"], rows

**Next:** look anything up in [Python reference 8: files, paths and formats](../learning/python-reference-08-files-paths-and-formats.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).